# 基线模型
前面准备完数据，现在可以开始建模了。我们就打算先使用简单的多元线性回归，不使用其他模型。因为本项目的核心目的不是做到非常好的效果，而是实操完整流程。

在开始前，我们先明确一下流程。首先要把前面准备好的数据拿过来完成preprocessing和statistical selection，这一步我们没有在前面处理干净，相当于数据准备的部分留了点小尾巴。

因为我们打算准备三组feature set:
* Set A: full features
* Set B: statistical selection
* Set C: manual engineered + statistical selection

而使用statistical selection就必须建立在数据处理彻底完成之前。而我们还必须要采用交叉验证，所以在真正开始之前，要先把前面的部分代码固定的.py文件里面来复用。不过没关系，我们这次跑的是baseline，只跑Set A full features

我们这一节的项结构就这么处理吧：
```
04_Modelling_baseline.ipynb
数据读取、导入
定义输入输出，即X和y
Scaling and One hot Encoding
划分验证集
模型训练
交叉验证
模型分析
```

## 数据准备
### 数据、地址与模块导入
第一部分，导入并准备数据。数据需要分成X和y，并进一步分成训练集和测试集。

值得一提的是，csv不保存数据格式，每次读取都会重新推断数据类型，所以必须手动重新处理一下MSSubClass的类型。这提醒我以后保存都用parquet比较好

In [162]:
from pathlib import Path

import pandas as pd

CURRENT_DIR = Path().cwd()
if CURRENT_DIR.name == "notebooks":
    #当前在notebooks/下面
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    #当前在根目录
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

df = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
df["MSSubClass"] = df["MSSubClass"].astype(str)

### 定义输入输出，即X和y
首先，我们直接跑一个线性回归模型，首先需要划分训练集。现在df已经读取到了没有进行过任何特征工程的数据集，这个数据集只填补了空缺值，并把部分有序类别变量直接编码成了离散的数值变量。

In [163]:
X = df.drop(columns="SalePrice")
X

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,HasGarage
0,60,RL,65.0,8450,Pave,NoAlley,1,Lvl,4,Inside,...,0,0,0,NoMisc,0,2,2008,WD,Normal,1
1,20,RL,80.0,9600,Pave,NoAlley,1,Lvl,4,FR2,...,0,0,0,NoMisc,0,5,2007,WD,Normal,1
2,60,RL,68.0,11250,Pave,NoAlley,2,Lvl,4,Inside,...,0,0,0,NoMisc,0,9,2008,WD,Normal,1
3,70,RL,60.0,9550,Pave,NoAlley,2,Lvl,4,Corner,...,0,0,0,NoMisc,0,2,2006,WD,Abnorml,1
4,60,RL,84.0,14260,Pave,NoAlley,2,Lvl,4,FR2,...,0,0,0,NoMisc,0,12,2008,WD,Normal,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446,60,RL,62.0,7917,Pave,NoAlley,1,Lvl,4,Inside,...,0,0,0,NoMisc,0,8,2007,WD,Normal,1
1447,20,RL,85.0,13175,Pave,NoAlley,1,Lvl,4,Inside,...,0,0,3,NoMisc,0,2,2010,WD,Normal,1
1448,70,RL,66.0,9042,Pave,NoAlley,1,Lvl,4,Inside,...,0,0,4,Shed,2500,5,2010,WD,Normal,1
1449,20,RL,68.0,9717,Pave,NoAlley,1,Lvl,4,Inside,...,0,0,0,NoMisc,0,4,2010,WD,Normal,1


In [164]:
y = df["SalePrice"]
y

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1446    175000
1447    210000
1448    266500
1449    142125
1450    147500
Name: SalePrice, Length: 1451, dtype: int64

### 划分验证集
X和y准备好以后，还需要分割训练集和验证集。分割可以使用sklearn.model_selection的现成函数train_test_split()

非常重要的一点，这个划分必须要提前做，而不能先做scaling和encoding再split，因为scaling会计算整体数据的scale，进而决定大小，先scaling再split就会导致泄露

In [165]:
from sklearn.model_selection import train_test_split, cross_validate

X_train, X_val, y_train, y_val = train_test_split(
    X,y,
    test_size=0.2, # 切分比例为20%
    random_state=42,# 种子设置为42
)

划分之后，再各自做处理。

### Scaling and One hot Encoding
但是至少X中的诸多类需要经过1-hot encoding才能进入模型，所以这里复用02_Cleaning_and_preprocessing的代码。四个数据子集都需要做。

y只有一列，所以处理特殊，不需要取出类别变量，也不需要1hot encoding

encoder实际上是可以理解为一个可学习的功能器件，在做编码的时候，需要根据一定的数据来构造转换规则。而这样，如果在trainset上学习一下，可以在training set 和validation set两个地方使用，而不能在validation上使用。因为validation模拟的是不能得到的数据，如果让validation自己根据自己来做转化，肯定效果好，但这样不符合真实假设。说白了，就是轻微的数据泄露。

还有一个问题，由于一列的列名都是相同的，因此不需要多次取列名或者标记是哪个子训练集的列名。

还有个问题，由于某些类的答案取值可能很多，可能不是所有的取值都完全掉进测试集，这种情况就会出现报错，这种报错可以忽略。

In [166]:
from sklearn.preprocessing import OneHotEncoder

#实例化一个encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 首先获取所有数据集的所有的类别变量列及其列名
# 先把训练集所有的类别变量列取出来
cat_columns = X_train.select_dtypes("str")
# 再取列名
cat_columns_name = cat_columns.columns.tolist()

# 再把验证集所有的类别变量列取出来
cat_columns_Xval = X_val[cat_columns_name]
# cat_columns_Xval = X_val.select_dtypes("str")


# 编码，给各个列都编码，编码验证集的时候应该使用训练集的参数来转换，因此不能带fit
# fit_transform就是先fit，再transform，也就是先根据数据集计算参数，如IQR和均值等，再转换
# 不能使用验证集自己的数据计算均值，应该使用训练集的数据计算参数，然后直接应用到验证集上
encoded_cat_Xtrain = encoder.fit_transform(cat_columns)
encoded_cat_Xval = encoder.transform(cat_columns_Xval)


# 把编码后的列表转化成Dataframe
encoded_cat_Xtrain_df = pd.DataFrame(
    encoded_cat_Xtrain,
    columns=encoder.get_feature_names_out(cat_columns_name),
    index=X_train.index
)

encoded_cat_Xval_df = pd.DataFrame(
    encoded_cat_Xval,
    columns=encoder.get_feature_names_out(cat_columns_name),
    index=X_val.index
)

# 以X_train为例看一下效果
encoded_cat_Xtrain_df

,MSSubClass_120,MSSubClass_160,MSSubClass_180,MSSubClass_190,MSSubClass_20,MSSubClass_30,MSSubClass_40,MSSubClass_45,MSSubClass_50,MSSubClass_60,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
1287,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
416,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1166,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
86,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1130,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1294,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
860,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


现在df中类别变量的一部分已经编码好了，我们把原来的类别变量直接drop，然后和编码好的拼在一起，这就是X了

In [167]:
# 把编码好的这些列和原来的数值列拼在一起
# 把验证集和测试集各自拼接到一起
X_train = pd.concat([X_train.drop(columns=cat_columns_name), encoded_cat_Xtrain_df], axis=1)
X_val = pd.concat([X_val.drop(columns=cat_columns_name), encoded_cat_Xval_df], axis=1)


# 以X_train为例展示
X_train

,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
1287,70.0,8400,1,4,1,5,5,1968,1968,168.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1035,69.0,9130,1,4,1,6,8,1966,2000,252.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
416,86.0,22420,2,4,1,6,6,1918,1950,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1166,35.0,4017,2,4,1,7,5,2006,2007,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
86,122.0,11911,3,4,1,6,5,2005,2005,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,61.0,9758,2,4,1,5,5,1971,1971,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1130,80.0,9600,1,4,1,6,5,1950,1950,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1294,92.0,10120,1,4,1,8,5,1994,1994,391.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
860,70.0,7931,1,4,1,5,5,1959,1959,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


这样就把类别变量都用1-hot encoding编码好了。可以看到由于有很多编码的变量，总列数来到了227列。这样X就准备好了，y我们也执行一下对数变换。这一步是三个Feature Set 共有的，都要执行。注意有两个

In [168]:
import numpy as np
y_train = np.log(y_train)
y_val = np.log(y_val)

然后我们执行预处理的剩下部分，做scaling，y不用做，分类变量已经1hot编码了也不用做。把X里面的数值变量拿出来做一下Scaling就可以了，继续复用02节的代码。我们采用RobustScaler。

同样需要分组对四个子集分别作。注意现在已经经过了1hot编码，所以所有的数据集都是纯数字的。我们需要拿出来原本就是纯数字的列来，所以要从df中取，而不能从X中取

In [169]:
from sklearn.preprocessing import RobustScaler
# 新建一个转换器对象
scaler = RobustScaler()
# 取出数值列
# num_features = X_train.select_dtypes("number").columns
num_features = df.drop("SalePrice", axis=1).select_dtypes("number").columns.tolist()

# 更新数值列
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_val[num_features] = scaler.transform(X_val[num_features])

## 模型训练

数据准备好以后，我们打算先跑一个最简单的线性回归模型。

In [170]:
from sklearn.linear_model import LinearRegression
LRmodel = LinearRegression()
LRmodel.fit(X_train, y_train)
print("Fit succeed")

Fit succeed


由于pycharm中的jupyter notebook有显示bug，这里额外添加了一句来抑制.fit()函数的输出。训练完直接预测，看看效果。注意看准确率是没有意义的，这个不是分类，是预测，预测价格差0.001都不行，所以准确率没意义。

In [171]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = LRmodel.predict(X_val)
print("R²：", r2_score(y_val, y_pred))
print("RMSE：", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE：", mean_absolute_error(y_val, y_pred))


R²： 0.8935148860170353
RMSE： 0.12558637060352157
MAE： 0.08307019966686445


模型解释了将近89%的波动，看起来还是非常有效的，RMSE表示预测误差的平方根，和y一个量纲，相当于“平均下来差了多少钱”，很有参考意义，当然，我们这个是取了log的，如果要是看真正的房价的话，再计算指数就可以了。


In [172]:
y_pred_inverted = np.exp(y_pred)
y_val_inverted = np.exp(y_val)
print("Mean(in dollar): ", np.mean(abs(y_val_inverted - y_val)))
print("RMSE(in dollar)：", np.sqrt(mean_squared_error(y_val_inverted, y_pred_inverted)))
print("MAE(in dollar)：", mean_absolute_error(y_val_inverted, y_pred_inverted))

Mean(in dollar):  179977.3632234067
RMSE(in dollar)： 20062.202549818852
MAE(in dollar)： 14006.099271775869


这就是说，平均下来，报价的误差在14000美元上下，方差在20000左右，这么看来误差就很大了。

## 交叉验证
我们的模型已经跑完一次了，接下来做一下交叉验证就可以了。这样整体baseline就算完全跑通了。交叉验证也可以直接用sklearn里面的现成函数，不过我们采用自动化程度比较低的KFold.split()来处理，这样可以理解的更清晰。等到正式固定代码的时候，我们再用自动化程度更高的函数

首先，我们把预处理封装一下。由于后面想要手写交叉验证，而交叉验证也会切分数据集，他将把数据集切分成n份，和百分比是一个意思，所以这个函数不应该同样切分数据集。输入应该是已经切分好的X_train, X_val, y_train, y_val，输出是处理好的X_train, X_val, y_train, y_val


In [173]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler

def preprocessing(X_train, X_val, y_train, y_val):
    # Scaling and 1-hot Encoding
    #实例化一个encoder
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    # 首先获取所有数据集的所有的类别变量列及其列名
    # 先把训练集所有的类别变量列取出来
    cat_columns = X_train.select_dtypes("str")
    cat_columns_name = cat_columns.columns.tolist()

    # 再把验证集所有的类别变量列取出来
    cat_columns_Xval = X_val[cat_columns_name]

    # 编码
    encoded_cat_Xtrain = encoder.fit_transform(cat_columns)
    encoded_cat_Xval = encoder.transform(cat_columns_Xval)

    # 把编码后的列表转化成Dataframe
    encoded_cat_Xtrain_df = pd.DataFrame(
        encoded_cat_Xtrain,
        columns=encoder.get_feature_names_out(cat_columns_name),
        index=X_train.index
    )

    encoded_cat_Xval_df = pd.DataFrame(
        encoded_cat_Xval,
        columns=encoder.get_feature_names_out(cat_columns_name),
        index=X_val.index
    )

    # 把验证集和测试集各自拼接到一起
    X_train = pd.concat([X_train.drop(columns=cat_columns_name), encoded_cat_Xtrain_df], axis=1)
    X_val = pd.concat([X_val.drop(columns=cat_columns_name), encoded_cat_Xval_df], axis=1)

    # 对数处理y
    y_train = np.log(y_train)
    y_val = np.log(y_val)

    #
    # 新建一个转换器对象
    scaler = RobustScaler()
    # 取出数值列
    num_features = X_train.select_dtypes("number").columns.tolist()

    # 更新数值列
    X_train[num_features] = scaler.fit_transform(X_train[num_features])
    X_val[num_features] = scaler.transform(X_val[num_features])

    return X_train, X_val, y_train, y_val


数据处理代码写好后，利用KFold.split()辅助划分

In [174]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

r2_score_list = []
rmse_list = []
mae_list = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(kf.split(X_train), start=1):
    # 利用给定的indexes, 完成本轮数据集划分
    # 得到的是每一个fold的训练集，因此列名一样，但是长度很短（行数少）
    X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    # 完成剩下的预处理
    X_train_fold, X_val_fold, y_train_fold, y_val_fold = preprocessing(X_train_fold, X_val_fold, y_train_fold, y_val_fold)

    # 训练模型
    LRmodel = LinearRegression()
    LRmodel.fit(X_train_fold, y_train_fold)

    # 添加分数
    y_pred_fold= LRmodel.predict(X_val_fold)
    r2_score_list.append(r2_score(y_val_fold, y_pred_fold))
    mae_list.append(mean_absolute_error(y_val_fold, y_pred_fold))
    rmse_list.append(np.sqrt(mean_squared_error(y_val_fold, y_pred_fold)))

    # 用于分析的分数
    # 测试集分数
    train_r2_score = np.mean(r2_score_list)

    print(f"Fold {fold} 已完成.")

Fold 1 已完成.
Fold 2 已完成.
Fold 3 已完成.
Fold 4 已完成.
Fold 5 已完成.


完成以后直接看看保存的分数。

In [175]:
print("R^2: ", np.array(r2_score_list).round(4))
print("RMSE: ", np.array(rmse_list).round(4))
print("MAE: ", np.array(mae_list).round(4))
# print(rmse_list)
# print(mae_list)

R^2:  [0.9169 0.8324 0.6873 0.8718 0.8083]
RMSE:  [0.0094 0.0149 0.0186 0.0119 0.0137]
MAE:  [0.0069 0.0085 0.0093 0.0081 0.008 ]


后面还要feature selection，我们目前的baseline是没有feature selection的，因此交叉验证也比较简单。如果需要做selection，那么每一个fold都要单独做scale

# 模型分析
模型分析这里，我们分析这些内容：
* 泛化表现
* Cross-validation 稳定性
* 第三层：Actual vs Predicted
* 第四层：Residuals
* 第五层：最大误差的数据点

不过首先，我们这里重新做一次交叉验证，这次采用自动化程度更高的Pipeline+cross_validate()，这也是方便后面封装代码。

In [176]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

# 定义数据集
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("str").columns

print("数据已经准备好.")
# 定义preprocessor
# 这个 处理器就是直接调用两种处理方法
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)
print("处理器已定义.")
# 把处理器和模型连接起来
# 数据管道由两方面构成，一个是数据处理器，一个是线性模型
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("LRmodel", LinearRegression()),
    ]
)
print("数据管道与模型已组装.")
# 做cross validate
# 定义Fold
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# 使用cross_validate计算
# 先定义scorings，是我们需要用到的评价指标
scorings={
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error",
        "MSE": "neg_mean_squared_error",
        "R2": "r2"
    }

scores = cross_validate(
    pipeline,
    X,
    np.log(y),
    cv=cv,
    scoring=scorings,
    return_train_score=True
)
print("交叉验证已完成.")

数据已经准备好.
处理器已定义.
数据管道与模型已组装.
交叉验证已完成.


这个`scores`是个字典，先前传入的指标就在里面，现在可以直接取出了，首先看一下key

In [181]:
print(scores.keys())

dict_keys(['fit_time', 'score_time', 'test_RMSE', 'train_RMSE', 'test_MAE', 'train_MAE', 'test_MSE', 'train_MSE', 'test_R2', 'train_R2'])


直接输出完整数据，干脆乘以-1，方便看

In [186]:
for key, value in scores.items():
    print(f"{key}: {-value}")

fit_time: [-0.22658277 -0.06127429 -0.06530619 -0.06142163 -0.07968473]
score_time: [-0.01475549 -0.01423073 -0.01385617 -0.01516581 -0.02510929]
test_RMSE: [0.12558637 0.23129887 0.13473137 0.14852255 0.12993759]
train_RMSE: [0.09530283 0.08913165 0.09237913 0.09407608 0.09693525]
test_MAE: [0.0830702  0.10205707 0.09036191 0.08617897 0.08820443]
train_MAE: [0.06689223 0.06370768 0.06453144 0.06620707 0.06691241]
test_MSE: [0.01577194 0.05349917 0.01815254 0.02205895 0.01688378]
train_MSE: [0.00908263 0.00794445 0.0085339  0.00885031 0.00939644]
test_R2: [-0.89351489 -0.67927731 -0.88396795 -0.85844817 -0.8996878 ]
train_R2: [-0.94405297 -0.94951183 -0.94673714 -0.94474394 -0.94024473]


我们来看RMSE, 首先能看出来，train_RMSE明显比test_RMSE小，小了将近10倍，这就是说明泛化性不好，存在严重的过拟合。

其次是，train_RMSE都差不多，相对比较稳定，但是test_RMSE却有一个明显过大的，第二个fold明显比其他fold大，这就说明可能第二个fold有一些很难预测的数据。同理R2也是，训练集上，基本都是94%左右，而测试集上（其实是验证集）也是第二个fold明显第很多，只有68%，说明有很多波动没有解释干净。

第三，RMSE比MAE一般而言大一些，MAE是线性的误差，而RMSE是平方误差，这就有趣了，对于表现不好的第二fold尤其明显，说明应该是少部分表现比较差的数据影响导致的。Fold 2其实值得单独研究

In [192]:
print("train_RMSE", -scores["train_RMSE"].mean().round(5),"±", scores["train_RMSE"].std().round(5))
print("test_RMSE", -scores["test_RMSE"].mean().round(5),"±", scores["test_RMSE"].std().round(5))
print("train_MAE", -scores["train_MAE"].mean().round(5),"±", scores["train_MAE"].std().round(5))
print("test_MAE", -scores["test_MAE"].mean().round(5),"±", scores["test_MAE"].std().round(5))
print("train_R2", -scores["train_R2"].mean().round(5),"±", scores["train_R2"].std().round(5))
print("test_R2", -scores["test_R2"].mean().round(5),"±", scores["test_R2"].std().round(5))

train_RMSE 0.09356 ± 0.00267
test_RMSE 0.15402 ± 0.0394
train_MAE 0.06565 ± 0.0013
test_MAE 0.08997 ± 0.0065
train_R2 -0.94506 ± 0.00306
test_R2 -0.84298 ± 0.08305
